In [2]:
# random_forest_test.ipynb - exploring setting up an RF model to predict ET
# Author: Archie Benn
# Date: 12-07-2026

# import libraries
import numpy as np
import pandas as pd
import optuna
import os
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error


In [3]:
# set seed for NumPy
np.random.seed(42)

# setup wd 
os.chdir('/home/ab/Dropbox/university/github/bbinf_project')
os.getcwd()


'/home/ab/Dropbox/university/github/bbinf_project'

In [4]:
# import data
df_14 = pd.read_csv("data/main/14_pre_processing/df_ml_ready.csv")
df_14.head()


,Unnamed: 0,Site_ID,Date,Latitude,Longitude,Site_age,Age_range,ET,LE,Lai_500m,...,Continent_Europe,Continent_North America,Cover_type_DBF,Cover_type_EBF,Cover_type_ENF,Cover_type_MF,Cover_type_OSH,Climate_zone_Dry,Climate_zone_Temperate,Climate_zone_Tropical
0,0,BE-Bra,2005-01-15,51.30761,4.51984,81,51-100,0.132589,3.83379,0.397561,...,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
1,1,BE-Bra,2005-01-16,51.30761,4.51984,81,51-100,0.052709,1.52237,0.386585,...,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
2,2,BE-Bra,2005-01-17,51.30761,4.51984,81,51-100,0.064565,1.85631,0.375610,...,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
3,3,BE-Bra,2005-01-18,51.30761,4.51984,81,51-100,0.105265,3.03227,0.364634,...,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
4,4,BE-Bra,2005-01-19,51.30761,4.51984,81,51-100,0.149545,4.30787,0.353659,...,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0


In [5]:
# set features and target
y = df_14['ET']

non_features_list = [
                     'Site_ID',       
                     'Latitude',
                     'Longitude',
                     'Age_range',
                     'ET',
                     'LE',
                     'P',                # have cumulative sum already
                     'Pa',               # dropped
                     'Date',
                     'Unnamed: 0',       # not sure where this came from
                     #'Site_age'         # leave out or keep in
                     ]


# drop the non features to get a features df
X = df_14.drop(columns=non_features_list)

len(y) == len(X)


True

In [6]:
y.head()


0    0.132589
1    0.052709
2    0.064565
3    0.105265
4    0.149545
Name: ET, dtype: float64

## Test RF model on one site


In [7]:
# set cross validation method to leave one group out
cv = LeaveOneGroupOut()

# set sites as groups to split by  
sites = df_14["Site_ID"] 


In [8]:
# optuna hyperparameter tuning on one site as a test before doing on all
# will allow me to reduce the search area for the larger loop later
def param_searcher(trial):

    # define search zone
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 450, step=100),
        "max_depth": trial.suggest_int("max_depth", 5, 50),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 20),
        "max_features": trial.suggest_float("max_features", 0.3, 1.0),
    }

    # set cross validation method to leave one group out
    cv = LeaveOneGroupOut()

    # set trial test sit
    test_site = "US-MMS"

    # run the leaveOneOut generator (cv) until the test index name == test site
    for train_idx, test_idx in cv.split(X, y, groups=sites):

        # if site name in test split == set test site
        if sites.iloc[test_idx].unique()[0] == test_site:
            break

    # set train/test split for features (X df) using indices from the leaveOneOute generator
    # train_idx is the row indices of the training rows (ie. non test site rows, opposite for test_idx)
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]

    # and same for train/test values of target/ET
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    # now actually setup the random forest
    # model params setup
    rf = RandomForestRegressor(

        # use parameters defined by optuna search (** expands dict)
        **params,
        random_state=42,
        # use all except one core
        n_jobs=-2)
    
    # and actually training the model
    rf.fit(X_train, y_train)

    # predictions
    preds = rf.predict(X_test)

    # calculate rmse and return
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    return rmse  


In [9]:
# setup optuna to minimise rmse during the runs
study = optuna.create_study(direction="minimize")

# no run optuna using the param searcher defined above set to minimise rmse
study.optimize(param_searcher, n_trials=10)


[I 2026-07-21 12:44:40,315] A new study created in memory with name: no-name-4d34b89b-ba19-4b9a-9acb-92bc58158f31
[I 2026-07-21 12:45:02,688] Trial 0 finished with value: 0.7114935966840646 and parameters: {'n_estimators': 250, 'max_depth': 25, 'min_samples_leaf': 3, 'max_features': 0.5211681324304638}. Best is trial 0 with value: 0.7114935966840646.
[W 2026-07-21 12:45:24,851] Trial 1 failed with parameters: {'n_estimators': 450, 'max_depth': 13, 'min_samples_leaf': 4, 'max_features': 0.6575649067793305} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/home/ab/micromamba/envs/bbinf_project/lib/python3.13/site-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
  File "/tmp/ipykernel_18198/625745886.py", line 44, in param_searcher
    rf.fit(X_train, y_train)
    ~~~~~~^^^^^^^^^^^^^^^^^^
  File "/home/ab/micromamba/envs/bbinf_project/lib/python3.13/site-packages/sklearn/base.py", line 1336, in wr

KeyboardInterrupt: 

### optuna results for RF model  
- ran on a few single held out sites to get a feel for params to use in main loop  

```
FI-Hyy:
[I 2026-07-15 09:35:28,288] Trial 7 finished with value: 0.3883278784490601 and parameters: {'n_estimators': 450, 'max_depth': 8, 'min_samples_leaf': 8, 'max_features': 0.7981815908116683}. Best is trial 7 with value: 0.3883278784490601.

CA-NS6:
[I 2026-07-15 09:41:45,367] Trial 6 finished with value: 0.3068583891853103 and parameters: {'n_estimators': 250, 'max_depth': 46, 'min_samples_leaf': 12, 'max_features': 0.33288864514664723}. Best is trial 6 with value: 0.3068583891853103.

IT-Cpz:
[I 2026-07-15 09:53:19,593] Trial 3 finished with value: 0.47599687105649824 and parameters: {'n_estimators': 50, 'max_depth': 9, 'min_samples_leaf': 10, 'max_features': 0.5805618584939788}. Best is trial 3 with value: 0.47599687105649824.

DE-Tha:
[I 2026-07-15 10:19:11,121] Trial 8 finished with value: 0.46392386288104803 and parameters: {'n_estimators': 250, 'max_depth': 6, 'min_samples_leaf': 10, 'max_features': 0.6483838076050964}. Best is trial 8 with value: 0.46392386288104803.

US-MMS:
[I 2026-07-15 10:27:03,777] Trial 7 finished with value: 0.7540318318542445 and parameters: {'n_estimators': 250, 'max_depth': 7, 'min_samples_leaf': 8, 'max_features': 0.9089800401389507}. Best is trial 7 with value: 0.7540318318542445.
```

- essentially, this is a very spread out set of parameters from optuna  
- as such, i will take a 'safe middle ground' approach for the final loop in `16_random_forest.py`   
- so will use this:

```
n_estimators=250, 
max_depth=8-10, 
min_samples_leaf=10,
 max_features=0.6
```